In [4]:
import os
import numpy as np
import librosa
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.utils import to_categorical


In [5]:
# ------------------------------
# 1️⃣ Define Dataset Path
# ------------------------------
data_path = r"C:\Users\Mohammed\Downloads\audio_speech_actors_01-24"


In [6]:

# ------------------------------
# 2️⃣ Feature Extraction Function (MFCC + reshape for CNN)
# ------------------------------
def extract_features_cnn(data_path, n_mfcc=40, max_len=174):
    features = []
    labels = []

    for root, dirs, files in os.walk(data_path):
        for file in files:
            if file.endswith(".wav"):
                file_path = os.path.join(root, file)
                try:
                    y, sr = librosa.load(file_path, sr=None)
                    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
                    
                    # Pad or truncate to max_len
                    if mfccs.shape[1] < max_len:
                        pad_width = max_len - mfccs.shape[1]
                        mfccs = np.pad(mfccs, pad_width=((0,0),(0,pad_width)), mode='constant')
                    else:
                        mfccs = mfccs[:, :max_len]

                    features.append(mfccs)
                    
                    emotion = int(file.split("-")[2])
                    labels.append(emotion)
                except Exception as e:
                    print("Error loading:", file_path)
                    continue

    X = np.array(features)
    y = np.array(labels)
    
    # Reshape for CNN: (samples, n_mfcc, max_len, 1)
    X = X[..., np.newaxis]
    return X, y

In [7]:
# ------------------------------
# 3️⃣ Extract Features
# ------------------------------
X, y = extract_features_cnn(data_path)
print("Features shape:", X.shape)
print("Labels shape:", y.shape)


Features shape: (1440, 40, 174, 1)
Labels shape: (1440,)


In [8]:
# ------------------------------
# 4️⃣ Label Encoding + One-hot
# ------------------------------
emotion_map = {1:"neutral", 2:"calm", 3:"happy", 4:"sad", 5:"angry", 6:"fearful", 7:"disgust", 8:"surprised"}
y_text = [emotion_map.get(label, "unknown") for label in y]

le = LabelEncoder()
y_encoded = le.fit_transform(y_text)
y_categorical = to_categorical(y_encoded)
print("First 10 one-hot encoded labels:\n", y_categorical[:10])


First 10 one-hot encoded labels:
 [[0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]]


In [9]:

# ------------------------------
# 5️⃣ Train/Test Split
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)


In [10]:

# ------------------------------
# 6️⃣ Build CNN Model
# ------------------------------
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(X.shape[1], X.shape[2], 1)),
    MaxPooling2D((2,2)),
    Dropout(0.3),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Dropout(0.3),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(le.classes_), activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\Mohammed\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 38, 172, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 19, 86, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 19, 86, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 17, 84, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 42, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 42, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 21504)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,752,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,772,488 (10.58 MB)

 Trainable params: 2,772,488 (10.58 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# ------------------------------
# 7️⃣ Train CNN
# ------------------------------
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1)


Epoch 1/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 301ms/step - accuracy: 0.1216 - loss: 20.1277 - val_accuracy: 0.0517 - val_loss: 2.0900
Epoch 2/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 391ms/step - accuracy: 0.1380 - loss: 2.0863 - val_accuracy: 0.1207 - val_loss: 2.0763
Epoch 3/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 358ms/step - accuracy: 0.1371 - loss: 2.0792 - val_accuracy: 0.1293 - val_loss: 2.0753
Epoch 4/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 400ms/step - accuracy: 0.1438 - loss: 2.0790 - val_accuracy: 0.1293 - val_loss: 2.0740
Epoch 5/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 299ms/step - accuracy: 0.1400 - loss: 2.0749 - val_accuracy: 0.0948 - val_loss: 2.0723
Epoch 6/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 328ms/step - accuracy: 0.1429 - loss: 2.0745 - val_accuracy: 0.1293 - val_loss: 2.0718
Epoch 7/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 312ms/step - accuracy: 0.1380 - loss: 2.0741 - val_accuracy: 0.1466 - val_loss: 2.0702
Epoch 8/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 312ms/step - accuracy: 0.1429 - loss: 2.0710 - val_acc

In [12]:

# ------------------------------
# 8️⃣ Evaluate
# ------------------------------
loss, acc = model.evaluate(X_test, y_test)
print("\nTest Accuracy:", acc)


9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - accuracy: 0.1111 - loss: 2.0669

Test Accuracy: 0.1111111119389534


In [13]:

# ------------------------------
# 9️⃣ Predict & Classification Report
y_pred = model.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

from sklearn.metrics import classification_report
print("\nClassification Report:\n", classification_report(y_true_labels, y_pred_labels, target_names=le.classes_))

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step 

Classification Report:
               precision    recall  f1-score   support

       angry       0.00      0.00      0.00        42
        calm       0.00      0.00      0.00        44
     disgust       0.11      0.97      0.21        32
     fearful       0.06      0.03      0.04        32
       happy       0.00      0.00      0.00        34
     neutral       0.00      0.00      0.00        20
         sad       0.00      0.00      0.00        39
   surprised       0.00      0.00      0.00        45

    accuracy                           0.11       288
   macro avg       0.02      0.12      0.03       288
weighted avg       0.02      0.11      0.03       288



C:\Users\Mohammed\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Mohammed\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Mohammed\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
